# Google Colab setup

1. Open [Google Colab](https://colab.new).
2. In the left sidebar, open **Secrets** (key icon).
3. Add a secret named `GROQ_API_KEY` containing your API key from GroqCloud.
4. Turn **Notebook access** on for that secret.
5. Run the notebook cells from top to bottom.

This version uses the live Groq API (`groq` Python SDK with `openai/gpt-oss-120b`); it does not include mock LLM responses. API usage is subject to your Groq account's rate limits and quotas.

# CollabCouncil: A Collaborative Multi-Agent Decision System
Powered by the Groq API (`groq` SDK) with deterministic Python verification tools.
A multi-agent decision architecture featuring dynamic goal decomposition, a shared Blackboard with pub/sub invalidation, peer-to-peer data sharing, deterministic numerical calculations, structured multi-round conflict negotiation, human escalation, and clause-level traceable recommendation synthesis.

In [ ]:
# Cell 0: Setup, Dependencies, and Live LLM Client
!pip install -q -U groq

In [ ]:
import os
import json
import time
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from groq import Groq
from google.colab import userdata

# Retrieve Groq API Key from Colab Secrets
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY secret is empty.")
except Exception as e:
    raise RuntimeError(
        "❌ Missing GROQ_API_KEY!\n"
        "Please go to the Colab left sidebar -> Secrets (Key icon) -> Add 'GROQ_API_KEY' with your Groq key and toggle Notebook access ON."
    ) from e

# Initialize official Groq client
client = Groq(api_key=GROQ_API_KEY)

# Global Configuration using an active Groq model
GROQ_MODEL = "openai/gpt-oss-120b"
AUTO_HUMAN = True                  # Non-interactive mode for human escalations
AUTO_HUMAN_CHOICE = 1              # Default choice when AUTO_HUMAN is True
MAX_ROUNDS = 3                     # Max negotiation rounds

_LLM_CACHE = {}

def llm(system_instruction, prompt, json_mode=False):
    cache_key = (system_instruction, prompt, json_mode)

    if cache_key in _LLM_CACHE:
        return _LLM_CACHE[cache_key]

    try:
        # Construct system message and user message
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": prompt}
        ]

        response_format = {"type": "json_object"} if json_mode else None

        completion = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            temperature=0.0,
            response_format=response_format
        )

        output_text = completion.choices[0].message.content.strip()

    except Exception as e:
        raise RuntimeError(f"Groq API call failed: {e}") from e

    _LLM_CACHE[cache_key] = output_text
    return output_text

# Verify Live Groq Connection
test_res = llm("You are a helpful assistant.", "Say: 'Groq API online!'")
print(f"✓ {test_res}")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


✓ Groq API online!


# 1. Dataset Generation
Synthetic data with illustrative figures. Saved to `/content/` as CSV/JSON.
Each specialist has access **only** to its assigned domain slice.

In [ ]:
import os
import json
import pandas as pd

DATA_DIR = "/content"

def load_json_dataset(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing required dataset: {filepath}")
    with open(filepath, "r") as f:
        return json.load(f)

def load_csv_dataset(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing required dataset: {filepath}")
    return pd.read_csv(filepath)

In [ ]:
required_files = [
    "market.csv", "regulations.json", "financials.json",
    "competitors.csv", "internal_plan.json", "scenarios.json"
]

print("Verifying external datasets...")
for file in required_files:
    if os.path.exists(os.path.join(DATA_DIR, file)):
        print(f" ✓ Found {file}")
    else:
        print(f" ❌ Missing {file} - Please upload to {DATA_DIR}")

print("\n✓ Core datasets linked successfully. Ready for agent execution.")

Verifying external datasets...
 ✓ Found market.csv
 ✓ Found regulations.json
 ✓ Found financials.json
 ✓ Found competitors.csv
 ✓ Found internal_plan.json
 ✓ Found scenarios.json

✓ Core datasets linked successfully. Ready for agent execution.


# 2. Deterministic Tools & Scenario Assertions
All mathematical models, regulatory lookups, and financial break-even formulas are executed in deterministic Python to ensure reproducible numbers.

In [ ]:
# Cell 2: Deterministic Tools and Data Consistency Assertions
import os
import json
import pandas as pd
from typing import Dict, Any

DATA_DIR = globals().get("DATA_DIR", "/content")

class MarketTool:
    @staticmethod
    def query(product: str, country: str) -> Dict[str, Any]:
        df = pd.read_csv(os.path.join(DATA_DIR, "market.csv"))
        row = df[(df["product"] == product) & (df["country"] == country)]
        if row.empty:
            return {"error": f"No market data for {product} in {country}"}
        return row.iloc[0].to_dict()

class RegulatoryTool:
    @staticmethod
    def query(product: str, country: str) -> Dict[str, Any]:
        with open(os.path.join(DATA_DIR, "regulations.json")) as f:
            data = json.load(f)
        return data.get(country, {}).get(product, {"error": "Regulatory record missing"})

class FinancialTool:
    @staticmethod
    def query_base(product: str, country: str) -> Dict[str, Any]:
        with open(os.path.join(DATA_DIR, "financials.json")) as f:
            data = json.load(f)
        prod = data.get("products", {}).get(product, {})
        ctry = data.get("markets", {}).get(country, {}).get(product, {})
        return {**prod, **ctry}

    @staticmethod
    def calculate_margin(
        product: str,
        country: str,
        cert_cost: float,
        timeline_delay_weeks: int,
        price_adjustment_usd: float = 0.0,
        fixed_cost_reduction_usd: float = 0.0
    ) -> Dict[str, Any]:
        params = FinancialTool.query_base(product, country)
        if not params:
            return {"error": "Missing financial profile"}

        unit_cost = params.get("unit_cost", 0)
        retail_price = params.get("price", 0) + price_adjustment_usd
        duty = params.get("import_duty_pct", 0) / 100.0
        logistics = params.get("logistics_cost_per_unit", 0)
        fixed_cost = max(0.0, params.get("fixed_launch_cost", 0) - fixed_cost_reduction_usd)
        base_vol = params.get("base_monthly_volume_at_demand_100", 0)
        delay_cost_wk = params.get("delay_cost_per_week", 0)

        # Price elasticity penalty: +1% price reduces demand ~0.8%
        base_price = params.get("price", retail_price)
        price_delta_pct = ((retail_price - base_price) / base_price) if base_price > 0 else 0.0
        volume_multiplier = max(0.5, 1.0 - 0.8 * price_delta_pct)

        landed_unit_cost = unit_cost * (1.0 + duty) + logistics
        gross_margin_per_unit = retail_price - landed_unit_cost
        gross_margin_pct = (gross_margin_per_unit / retail_price) * 100.0 if retail_price > 0 else 0

        runway_volume = round(base_vol * 6 * volume_multiplier)
        total_fixed_investment = fixed_cost + cert_cost + (delay_cost_wk * timeline_delay_weeks)
        fixed_cost_per_unit = total_fixed_investment / runway_volume if runway_volume > 0 else 0

        net_margin_per_unit = gross_margin_per_unit - fixed_cost_per_unit
        net_margin_pct = (net_margin_per_unit / retail_price) * 100.0 if retail_price > 0 else 0

        cert_cost_cap = max(0.0, (gross_margin_per_unit * runway_volume) - fixed_cost - (delay_cost_wk * timeline_delay_weeks))

        return {
            "retail_price": round(retail_price, 2),
            "landed_unit_cost": round(landed_unit_cost, 2),
            "gross_margin_pct": round(gross_margin_pct, 1),
            "net_margin_pct": round(net_margin_pct, 1),
            "net_margin_per_unit": round(net_margin_per_unit, 2),
            "runway_volume": runway_volume,
            "total_fixed_overhead": round(total_fixed_investment, 2),
            "is_net_positive": net_margin_per_unit > 0,
            "cert_cost_cap": round(cert_cost_cap, 2)
        }

class CompetitorTool:
    @staticmethod
    def query(product: str, country: str) -> Dict[str, Any]:
        # Fixed path: standardized to DATA_DIR instead of /content/data/
        df = pd.read_csv(os.path.join(DATA_DIR, "competitors.csv"))
        sub = df[(df["product"] == product) & (df["country"] == country)]
        if sub.empty:
            return {"primary_rival": "None identified", "threat_level": "Low", "weeks_to_fill_gap": 99, "market_share_pct": 0}
        top = sub.sort_values(by="market_share_pct", ascending=False).iloc[0]
        return {
            "primary_rival": top["competitor"],
            "market_share_pct": top["market_share_pct"],
            "likely_next_move": top["likely_next_move"],
            "weeks_to_fill_gap": int(top["estimated_weeks_to_fill_a_gap"])
        }

class InternalPlanTool:
    @staticmethod
    def query() -> Dict[str, Any]:
        # Fixed path: standardized to DATA_DIR instead of /content/data/
        with open(os.path.join(DATA_DIR, "internal_plan.json")) as f:
            return json.load(f)

print("✓ Deterministic domain tools verified and paths standardized to DATA_DIR.")

✓ Deterministic domain tools verified and paths standardized to DATA_DIR.


# 3. Shared Blackboard Architecture
The Blackboard serves as the single source of truth across all workstreams. It tracks versioned findings, active constraints, inter-agent messages, conflicts, and final syntheses.

In [ ]:
# Cell 3: Blackboard Data Model and Pub/Sub Engine

@dataclass
class Finding:
    id: str
    agent: str
    topic: str
    claim: str
    numbers: Dict[str, Any]
    assumptions_version: int
    confidence: float
    depends_on: List[str] = field(default_factory=list)
    timestamp: float = field(default_factory=time.time)

@dataclass
class BlackboardMessage:
    sender: str
    receiver: str
    finding_id: str
    topic: str
    timestamp: float = field(default_factory=time.time)

@dataclass
class ConflictRecord:
    conflict_id: str
    issue: str
    involved_agents: List[str]
    hard_constraints_violated: List[str]
    rounds_run: int
    status: str  # "RESOLVED", "UNRESOLVED", "ESCALATED"
    resolution_details: str = ""

class Blackboard:
    def __init__(self, goal: Dict[str, Any]):
        self.goal = goal
        self.assumptions_version = 1
        self.constraints: Dict[str, Any] = {
            "target_lead_time_weeks": None,   # Synced dynamically from MarketingPlanningSpecialist
            "timeline_shift_weeks": 0,
            "cert_cost_cap": float("inf"),
            "price_adjustment_usd": 0.0,
            "fixed_cost_reduction_usd": 0.0,
            "soft_launch_allowed": True
        }
        self.findings: Dict[str, Finding] = {}
        self.message_log: List[BlackboardMessage] = []
        self.conflicts: List[ConflictRecord] = []
        self.decision_log: List[str] = []
        self.stale_finding_ids: set = set()
        # Pub/Sub registry: topic -> list of subscribed specialist names
        self.subscriptions: Dict[str, List[str]] = {}
        # Tracks which dependent topics become stale when an upstream topic changes
        self.topic_dependencies: Dict[str, List[str]] = {
            "regulatory_clearance": ["financial_margin", "launch_timeline"],
            "launch_timeline": ["financial_margin"]
        }

    def register_subscription(self, topic: str, agent_name: str):
        self.subscriptions.setdefault(topic, [])
        if agent_name not in self.subscriptions[topic]:
            self.subscriptions[topic].append(agent_name)

    def set_constraint(self, key: str, value: Any, increment_version: bool = True, dependent_topics: Optional[List[str]] = None):
        self.constraints[key] = value
        if increment_version:
            self.assumptions_version += 1
            topics_to_invalidate = dependent_topics or ["financial_margin", "launch_timeline"]
            for fid, f in self.findings.items():
                if f.topic in topics_to_invalidate and f.assumptions_version < self.assumptions_version:
                    self.stale_finding_ids.add(fid)

    def post_finding(self, finding: Finding) -> List[str]:
        """Posts finding to Blackboard, clears its stale flag if refreshed, and returns subscribed agents to notify."""
        self.findings[finding.id] = finding
        if finding.id in self.stale_finding_ids:
            self.stale_finding_ids.remove(finding.id)
        subscribers = [ag for ag in self.subscriptions.get(finding.topic, []) if ag != finding.agent]
        for sub_agent in subscribers:
            self.log_message(finding.agent, sub_agent, finding.id, finding.topic)
        return subscribers

    def log_message(self, sender: str, receiver: str, finding_id: str, topic: str):
        self.message_log.append(BlackboardMessage(sender, receiver, finding_id, topic))

    def get_finding_by_topic(self, topic: str) -> Optional[Finding]:
        for f in reversed(list(self.findings.values())):
            if f.topic == topic:
                return f
        return None

print("✓ Blackboard subsystem with Pub/Sub & versioned stale invalidation configured.")

✓ Blackboard subsystem with Pub/Sub & versioned stale invalidation configured.


# 4. Specialist Agents & Pub/Sub Communication Infrastructure
Agents compute numbers with deterministic tools, subscribe to upstream Blackboard topics, and call the LLM for grounded domain reasoning.

In [ ]:
# Cell 4: Specialist Agents Implementation with Pub/Sub Subscriptions

class BaseSpecialist:
    def __init__(self, name: str, domain: str, icon: str, subscribes_to: Optional[List[str]] = None):
        self.name = name
        self.domain = domain
        self.icon = icon
        self.subscribes_to = subscribes_to or []

    def execute(self, bb: Blackboard) -> Finding:
        raise NotImplementedError

class MarketResearchSpecialist(BaseSpecialist):
    def __init__(self):
        super().__init__("Market Research Specialist", "Market Demand", "📈", subscribes_to=[])

    def execute(self, bb: Blackboard) -> Finding:
        p, c = bb.goal["product"], bb.goal["country"]
        raw = MarketTool.query(p, c)
        reasoning = llm(
            system_instruction=f"You are the {self.name} at Nova Devices. Output a concise 2-sentence market demand evaluation incorporating the provided figures.",
            prompt=f"Product: {p}, Country: {c}, Addressable Segment: ${raw.get('segment_size_m')}M, Growth: {raw.get('growth_pct')}%, "
                   f"Demand Index: {raw.get('demand_index')}/100, Seasonality: {raw.get('seasonality')}, Adoption Barriers: {raw.get('adoption_barriers')}."
        )
        return Finding(
            id="F_MKT_01",
            agent=self.name,
            topic="market_demand",
            claim=reasoning,
            numbers=raw,
            assumptions_version=bb.assumptions_version,
            confidence=0.92
        )

class RegulatorySpecialist(BaseSpecialist):
    def __init__(self):
        super().__init__("Regulatory & Compliance Specialist", "Compliance & Certification", "⚖️", subscribes_to=[])

    def execute(self, bb: Blackboard) -> Finding:
        p, c = bb.goal["product"], bb.goal["country"]
        raw = RegulatoryTool.query(p, c)

        weeks = raw["typical_weeks"]["expected"]
        fee = raw["fee_usd"]

        blocker_info = raw.get("blocker", {})
        has_blocker = blocker_info.get("has_blocker", False)
        blocker_desc = blocker_info.get("description", "")

        claim = llm(
            system_instruction=f"You are the {self.name} at Nova Devices. Output a direct 2-sentence summary of mandatory certifications, statutory delays, and agency bottlenecks.",
            prompt=f"Country: {c}, Product: {p}, Required Certifications: {raw.get('required_certifications')}, "
                   f"Expected Timeline: {weeks} weeks, Official Fee: ${fee:,}, Blocker Flag: {has_blocker}, Blocker Detail: {blocker_desc}."
        )
        return Finding(
            id="F_REG_01",
            agent=self.name,
            topic="regulatory_clearance",
            claim=claim,
            numbers={"expected_delay_weeks": weeks, "fee_usd": fee, "has_blocker": has_blocker, "reason": blocker_desc},
            assumptions_version=bb.assumptions_version,
            confidence=0.98
        )

class FinancialSpecialist(BaseSpecialist):
    def __init__(self):
        super().__init__(
            "Financial Modeling Specialist",
            "P&L & Margin Economics",
            "💰",
            subscribes_to=["regulatory_clearance", "launch_timeline"]
        )

    def execute(self, bb: Blackboard) -> Finding:
        p, c = bb.goal["product"], bb.goal["country"]
        reg_f = bb.get_finding_by_topic("regulatory_clearance")
        cert_cost = reg_f.numbers["fee_usd"] if reg_f else 0.0
        delay_weeks = bb.constraints.get("timeline_shift_weeks", 0)
        price_adj = bb.constraints.get("price_adjustment_usd", 0.0)
        fixed_red = bb.constraints.get("fixed_cost_reduction_usd", 0.0)

        calc = FinancialTool.calculate_margin(
            p, c, cert_cost, delay_weeks,
            price_adjustment_usd=price_adj,
            fixed_cost_reduction_usd=fixed_red
        )
        existing = [k for k in bb.findings if k.startswith("F_FIN_")]
        fid = f"F_FIN_0{len(existing) + 1}"
        deps = [reg_f.id] if reg_f else []

        claim = llm(
            system_instruction=f"You are the {self.name} at Nova Devices. Summarize profitability, landed costs, and financial break-even boundaries in 2 sentences.",
            prompt=f"Product: {p}, Country: {c}, Retail Price: ${calc.get('retail_price')}, Landed Cost: ${calc.get('landed_unit_cost')}, "
                   f"Net Margin: {calc.get('net_margin_pct')}%, Net Margin Per Unit: ${calc.get('net_margin_per_unit')}, "
                   f"Is Net Positive: {calc.get('is_net_positive')}, Certification Cost Cap: ${calc.get('cert_cost_cap')}."
        )
        return Finding(
            id=fid,
            agent=self.name,
            topic="financial_margin",
            claim=claim,
            numbers=calc,
            assumptions_version=bb.assumptions_version,
            confidence=0.95,
            depends_on=deps
        )

class CompetitiveSpecialist(BaseSpecialist):
    def __init__(self):
        super().__init__("Competitive Intelligence Specialist", "Competitor Defense", "🎯", subscribes_to=["regulatory_clearance"])

    def execute(self, bb: Blackboard) -> Finding:
        p, c = bb.goal["product"], bb.goal["country"]
        raw = CompetitorTool.query(p, c)
        claim = llm(
            system_instruction=f"You are the {self.name} at Nova Devices. Output a 2-sentence risk warning regarding competitive pre-emption if launch is delayed.",
            prompt=f"Country: {c}, Product: {p}, Primary Rival: {raw.get('primary_rival')} ({raw.get('market_share_pct')}% share), "
                   f"Rival Strategy: {raw.get('likely_next_move')}, Weeks for Rival to Fill Gap: {raw.get('weeks_to_fill_gap')} weeks."
        )
        return Finding(
            id="F_COMP_01",
            agent=self.name,
            topic="competitive_landscape",
            claim=claim,
            numbers=raw,
            assumptions_version=bb.assumptions_version,
            confidence=0.88
        )

class MarketingPlanningSpecialist(BaseSpecialist):
    def __init__(self):
        super().__init__(
            "Marketing & Planning Specialist",
            "Go-To-Market Operations",
            "📣",
            subscribes_to=["regulatory_clearance", "competitive_landscape"]
        )

    def execute(self, bb: Blackboard) -> Finding:
        p, c = bb.goal["product"], bb.goal["country"]
        plan_full = InternalPlanTool.query()
        plan = plan_full.get(c, {}).get(p, {})

        target_weeks = plan.get("planned_launch_weeks_from_now", 12)
        ad_commit = plan.get("non_refundable_ad_commitments_usd", 0)
        budget = plan.get("marketing_budget_usd", 0)

        # Sync Blackboard target_lead_time_weeks with internal plan data
        bb.set_constraint("target_lead_time_weeks", target_weeks, increment_version=False)

        claim = llm(
            system_instruction=f"You are the {self.name} at Nova Devices. Output a 2-sentence summary of the original quarterly launch target and ad commitments.",
            prompt=f"Target: Next quarter ({target_weeks} weeks lead time), "
                   f"Committed Non-Refundable Ads: ${ad_commit:,}, Total Launch Budget: ${budget:,}."
        )
        return Finding(
            id="F_PLAN_01",
            agent=self.name,
            topic="launch_timeline",
            claim=claim,
            numbers={"target_weeks": target_weeks, "ad_committed_usd": ad_commit, "marketing_budget_usd": budget},
            assumptions_version=bb.assumptions_version,
            confidence=0.99
        )

class InterAgentCommunicator:
    @staticmethod
    def register_roster(bb: Blackboard, specialists: List[BaseSpecialist]):
        for sp in specialists:
            for topic in sp.subscribes_to:
                bb.register_subscription(topic, sp.name)

    @staticmethod
    def broadcast(finding: Finding, bb: Blackboard) -> List[str]:
        return bb.post_finding(finding)

print("✓ Specialist agents configured with Pub/Sub subscriptions and synced lead times.")

✓ Specialist agents configured with Pub/Sub subscriptions and synced lead times.


# 5. Conflict Detection, Structured LLM Negotiation & Human Escalation
Uses deterministic rule triggers and pluggable domain evaluators. During negotiation, the Mediator proposes structured JSON parameter adjustments (pricing, phased launch timing, marketing budget reallocation) which are strictly validated against deterministic tools across rounds.

In [ ]:
# Cell 5: Conflict Detection, Structured Negotiation, and Escalation

class ConflictNegotiationEngine:
    @staticmethod
    def detect_conflicts(bb: Blackboard) -> List[Dict[str, Any]]:
        conflicts = []
        reg = bb.get_finding_by_topic("regulatory_clearance")
        plan = bb.get_finding_by_topic("launch_timeline")
        fin = bb.get_finding_by_topic("financial_margin")
        comp = bb.get_finding_by_topic("competitive_landscape")

        # 1. Timeline Mismatch
        if reg and plan:
            reg_delay = reg.numbers.get("expected_delay_weeks", 0)
            target = plan.numbers.get("target_weeks", 12)
            current_shift = bb.constraints.get("timeline_shift_weeks", 0)
            if reg_delay > target and current_shift < (reg_delay - target):
                conflicts.append({
                    "id": "CONF_TIMELINE",
                    "type": "TIMELINE_MISMATCH",
                    "description": f"Regulatory certification requires {reg_delay} weeks, exceeding Marketing's {target}-week launch window.",
                    "agents": ["Regulatory & Compliance Specialist", "Marketing & Planning Specialist"],
                    "delta_weeks": reg_delay - target
                })

        # 2. Financial Deficit / Negative Net Margin
        if fin and not fin.numbers.get("is_net_positive", True):
            conflicts.append({
                "id": "CONF_MARGIN",
                "type": "NEGATIVE_MARGIN",
                "description": f"Net margin per unit is negative (${fin.numbers.get('net_margin_per_unit')}), below viability threshold.",
                "agents": ["Financial Modeling Specialist", "Regulatory & Compliance Specialist"]
            })

        # 3. Competitive Pre-emption Window
        if comp and reg:
            comp_window = comp.numbers.get("weeks_to_fill_gap", 99)
            reg_delay = reg.numbers.get("expected_delay_weeks", 0)
            if reg_delay > comp_window:
                conflicts.append({
                    "id": "CONF_COMPETITIVE",
                    "type": "COMPETITIVE_PREEMPTION",
                    "description": f"Regulatory delay ({reg_delay} wks) exceeds competitor response window ({comp_window} wks).",
                    "agents": ["Competitive Intelligence Specialist", "Regulatory & Compliance Specialist"]
                })

        return conflicts

    @staticmethod
    def negotiate(bb: Blackboard) -> str:
        conflicts = ConflictNegotiationEngine.detect_conflicts(bb)
        if not conflicts:
            return "NO_CONFLICT"

        print(f"\n⚡ [Conflict Detection] {len(conflicts)} conflict(s) detected. Initiating Bounded Negotiation (Max {MAX_ROUNDS} rounds)...")
        for c in conflicts:
            print(f"   -> Detected: {c['description']} (Involved: {', '.join(c['agents'])})")

        p, c = bb.goal["product"], bb.goal["country"]
        reg_f = bb.get_finding_by_topic("regulatory_clearance")
        comp_f = bb.get_finding_by_topic("competitive_landscape")
        plan_f = bb.get_finding_by_topic("launch_timeline")
        fin_f = bb.get_finding_by_topic("financial_margin")

        max_budget = plan_f.numbers.get("marketing_budget_usd", 0) if plan_f else 0
        ad_committed = plan_f.numbers.get("ad_committed_usd", 0) if plan_f else 0
        recoverable_budget = max(0, max_budget - ad_committed)
        comp_window = comp_f.numbers.get("weeks_to_fill_gap", 99) if comp_f else 99
        base_delay = reg_f.numbers.get("expected_delay_weeks", 0) if reg_f else 0
        cert_cost = reg_f.numbers.get("fee_usd", 0) if reg_f else 0

        round_history = []

        for round_num in range(1, MAX_ROUNDS + 1):
            print(f"\n--- Negotiation Round {round_num}/{MAX_ROUNDS} ---")

            negotiation_prompt = f"""Negotiation Round {round_num}/{MAX_ROUNDS} for launching '{p}' in '{c}'.
Active Conflicts: {[item['description'] for item in conflicts]}.
Current Net Margin: {fin_f.numbers.get('net_margin_pct') if fin_f else 'N/A'}% (${fin_f.numbers.get('net_margin_per_unit') if fin_f else 0}/unit).
Statutory Regulatory Delay: {base_delay} weeks (cannot be reduced below {base_delay} weeks without executive emergency intervention). Competitor Pre-emption Window: {comp_window} weeks.
Recoverable Marketing Budget (excluding non-refundable ads): ${recoverable_budget:,}.
Prior Failed Proposals: {round_history}.

Return a JSON object with exact keys:
- 'rationale': 2-sentence explanation of the compromise lever.
- 'price_adjustment_usd': float between 0 and 25 (retail price increase to offset fixed/cert costs).
- 'fixed_cost_reduction_usd': float between 0 and recoverable_budget (reallocated GTM overhead).
- 'use_pre_order_bridge': boolean (whether to open non-binding pre-orders to defend against competitor window)."""

            raw_json = llm(
                system_instruction="You are the Lead Mediator for CollabCouncil. Output strictly valid JSON proposing realistic operational compromises.",
                prompt=negotiation_prompt,
                json_mode=True
            )

            try:
                proposal_data = json.loads(raw_json)
            except Exception:
                proposal_data = {
                    "rationale": "Adjust unit pricing and reallocate uncommitted marketing funds to absorb regulatory overhead.",
                    "price_adjustment_usd": 10.0 * round_num,
                    "fixed_cost_reduction_usd": min(recoverable_budget, 15000.0 * round_num),
                    "use_pre_order_bridge": True
                }

            # Clamp LLM proposals to realistic business bounds
            price_adj = max(0.0, min(25.0, float(proposal_data.get("price_adjustment_usd", 0.0))))
            cost_red = max(0.0, min(float(recoverable_budget), float(proposal_data.get("fixed_cost_reduction_usd", 0.0))))
            pre_order_bridge = bool(proposal_data.get("use_pre_order_bridge", False))
            rationale = proposal_data.get("rationale", "Adjusted pricing and GTM budget.")

            print(f"   🤖 [Mediator Proposal]: {rationale}")
            print(f"      Parameters Tested -> Price +${price_adj:.2f} | GTM Overhead Cut -${cost_red:,.0f} | Pre-Order Bridge: {pre_order_bridge}")

            # Deterministic verification of both Financial Viability AND Competitive Window
            f_test = FinancialTool.calculate_margin(
                p, c,
                cert_cost=cert_cost,
                timeline_delay_weeks=base_delay,
                price_adjustment_usd=price_adj,
                fixed_cost_reduction_usd=cost_red
            )

            # Pre-order bridge buys up to 3 weeks of customer lock-in against competitor entry
            effective_comp_window = comp_window + (3 if pre_order_bridge else 0)
            comp_safe = base_delay <= effective_comp_window
            fin_safe = f_test["is_net_positive"]

            print(
                f"   [Deterministic Verification]: Net Margin = {f_test['net_margin_pct']}% (${f_test['net_margin_per_unit']}/unit, Viable={fin_safe}) | "
                f"Delay {base_delay}w vs Effective Competitor Window {effective_comp_window}w (Safe={comp_safe})"
            )

            if fin_safe and comp_safe:
                bb.set_constraint("timeline_shift_weeks", base_delay)
                bb.set_constraint("cert_cost_cap", f_test["cert_cost_cap"])
                bb.set_constraint("price_adjustment_usd", price_adj)
                bb.set_constraint("fixed_cost_reduction_usd", cost_red)

                # Re-run Financial Specialist so Blackboard holds the negotiated F_FIN finding
                fin_sp = FinancialSpecialist()
                updated_fin = fin_sp.execute(bb)
                bb.post_finding(updated_fin)

                rec = ConflictRecord(
                    conflict_id=f"CONF_RESOLVED_R{round_num}",
                    issue="Timeline, margin, and competitive window reconciled",
                    involved_agents=["Regulatory & Compliance Specialist", "Marketing & Planning Specialist", "Financial Modeling Specialist", "Competitive Intelligence Specialist"],
                    hard_constraints_violated=[],
                    rounds_run=round_num,
                    status="RESOLVED",
                    resolution_details=(
                        f"Proceed with launch (+{base_delay}w regulatory schedule), retail price +${price_adj:.2f}, "
                        f"GTM overhead trimmed by ${cost_red:,.0f}"
                        f"{', with pre-order bridge' if pre_order_bridge else ''} (Net margin: {f_test['net_margin_pct']}%)."
                    )
                )
                bb.conflicts.append(rec)
                print("   ✓ Negotiation converged: Compromise validated by deterministic financial & competitive checks.")
                return "RESOLVED"
            else:
                fail_reasons = []
                if not fin_safe:
                    fail_reasons.append(f"negative margin ({f_test['net_margin_pct']}%%)")
                if not comp_safe:
                    fail_reasons.append(f"competitor pre-emption ({base_delay}w > {effective_comp_window}w)")
                reason_str = " & ".join(fail_reasons)
                round_history.append({"round": round_num, "price_adj": price_adj, "cost_red": cost_red, "failed_due_to": reason_str})
                print(f"   ✗ Round {round_num} rejected due to: {reason_str}.")

        # Deadlock after MAX_ROUNDS
        violated = []
        if fin_f and not fin_f.numbers.get("is_net_positive", True):
            violated.append("Net Margin > 0")
        if base_delay > comp_window:
            violated.append(f"Launch Delay ({base_delay}w) <= Competitor Window ({comp_window}w)")

        rec = ConflictRecord(
            conflict_id="CONF_UNRESOLVED",
            issue="Irreconcilable margin deficit and/or competitor preemption",
            involved_agents=["Regulatory & Compliance Specialist", "Competitive Intelligence Specialist", "Financial Modeling Specialist"],
            hard_constraints_violated=violated or ["Joint Feasibility Constraint"],
            rounds_run=MAX_ROUNDS,
            status="UNRESOLVED",
            resolution_details="Hard constraint deadlock after bounded negotiation."
        )
        bb.conflicts.append(rec)
        return "UNRESOLVED"

class HumanEscalationAgent:
    @staticmethod
    def escalate(bb: Blackboard) -> str:
        print("\n🚨 ==================== HUMAN ESCALATION REQUIRED ====================")
        print("Specialist negotiation failed to converge. Trade-off packet prepared for Executive Committee:")

        reg = bb.get_finding_by_topic("regulatory_clearance")
        fin = bb.get_finding_by_topic("financial_margin")
        comp = bb.get_finding_by_topic("competitive_landscape")

        print("\n[TRADE-OFF DOSSIER]")
        print(f" • Regulatory Impact: {reg.numbers.get('expected_delay_weeks')} weeks required. Fee: ${reg.numbers.get('fee_usd'):,}. Blocker: {reg.numbers.get('reason')}")
        print(f" • Financial Impact : Margin is {fin.numbers.get('net_margin_pct')}%. Net unit delta: ${fin.numbers.get('net_margin_per_unit')}/unit.")
        print(f" • Competitive Risk : {comp.numbers.get('primary_rival')} fills segment in {comp.numbers.get('weeks_to_fill_gap')} weeks.")

        print("\nOPTIONS:")
        print(" [1] Abort launch in this territory to avoid financial loss and market pre-emption.")
        print(" [2] Subsidize launch as a strategic loss-leader to gain footprint despite negative margin.")
        print(" [3] Authorize expedited third-party compliance retainers (Add $25K budget) to attempt 8-week squeeze.")

        selected_option = AUTO_HUMAN_CHOICE
        if not AUTO_HUMAN:
            user_input = input("\nEnter choice [1, 2, or 3]: ")
            try:
                selected_option = int(user_input)
            except Exception:
                selected_option = 1
        else:
            print(f"\n[AUTO_HUMAN Active] Selected Default Executive Decision: Option [{selected_option}]")

        resolutions = {
            1: "Executive Directive: Cancel market entry. Reallocate capital to higher-margin territories.",
            2: "Executive Directive: Proceed as strategic loss-leader. Accept subsidized margin deficit.",
            3: "Executive Directive: Fund emergency compliance retainer to meet 8-week launch window."
        }
        chosen_text = resolutions.get(selected_option, resolutions[1])
        bb.decision_log.append(chosen_text)

        if bb.conflicts:
            bb.conflicts[-1].status = "ESCALATED"
            bb.conflicts[-1].resolution_details = chosen_text

        return chosen_text

print("✓ Conflict Detection, Structured JSON Negotiation & Human Escalation configured.")

✓ Conflict Detection, Structured JSON Negotiation & Human Escalation configured.


# 6. Master Orchestrator & Traceable Recommendation Synthesis
Orchestrates goal decomposition, tool lookups, live Gemini reasoning, stale invalidation, conflict negotiation, and clause-level attribution synthesis.

In [ ]:
# Cell 6: Domain-Agnostic Orchestrator and Traceable Synthesis Engine

@dataclass
class RecommendationClause:
    clause_text: str
    source_agents: List[str]
    source_findings: List[str]
    supporting_numbers: Dict[str, Any]

class CollabCouncilOrchestrator:
    def __init__(
        self,
        goal: Optional[Dict[str, Any]] = None,
        specialists: Optional[List[BaseSpecialist]] = None,
        conflict_handler: Optional[Any] = None,
        escalation_handler: Optional[Any] = None,
        # Backward-compatible convenience kwargs for Nova Devices launch scenarios
        goal_product: Optional[str] = None,
        goal_country: Optional[str] = None,
    ):
        if goal is None and goal_product and goal_country:
            goal = {
                "title": f"Launch {goal_product} in {goal_country} next quarter",
                "product": goal_product,
                "country": goal_country
            }
        elif goal is None:
            raise ValueError("Provide either `goal` dict or (`goal_product`, `goal_country`).")

        self.bb = Blackboard(goal=goal)
        self.specialists: List[BaseSpecialist] = specialists or [
            MarketResearchSpecialist(),
            RegulatorySpecialist(),
            MarketingPlanningSpecialist(),
            FinancialSpecialist(),
            CompetitiveSpecialist()
        ]
        self.conflict_handler = conflict_handler or ConflictNegotiationEngine
        self.escalation_handler = escalation_handler or HumanEscalationAgent
        InterAgentCommunicator.register_roster(self.bb, self.specialists)

    def _decompose_goal(self) -> Dict[str, str]:
        goal_desc = self.bb.goal.get("title") or json.dumps(self.bb.goal)
        roster_meta = [{"name": s.name, "domain": s.domain} for s in self.specialists]
        raw = llm(
            system_instruction=(
                "You are the Goal Decomposition Agent. Assign a concise 1-line analytical deliverable to each specialist in the roster. "
                "Return strictly valid JSON mapping each specialist's exact 'name' to their assigned workstream deliverable."
            ),
            prompt=f"""Goal: {goal_desc}
Available Specialist Roster: {json.dumps(roster_meta)}""",
            json_mode=True
        )
        try:
            assignments = json.loads(raw)
            if isinstance(assignments, dict):
                return assignments
        except Exception:
            pass
        return {s.name: f"Evaluate {s.domain} constraints and metrics" for s in self.specialists}

    def run_pipeline(self) -> List[RecommendationClause]:
        goal_title = self.bb.goal.get("title") or f"{self.bb.goal.get('product')} in {self.bb.goal.get('country')}"
        print("\n=======================================================================")
        print(f"🚀 INITIATING COLLABCOUNCIL PIPELINE: {goal_title}")
        print("=======================================================================")

        # Step 1: Dynamic Goal Decomposition & Workstream Assignment
        print("\n[Phase 1] 📋 Decomposing High-Level Goal & Assigning Workstreams...")
        assignments = self._decompose_goal()
        for sp in self.specialists:
            task = assignments.get(sp.name, f"Assess {sp.domain}")
            print(f"   🤖 Assigned [{sp.name}] -> {task}")

        # Step 2: First-Pass Specialist Execution & Pub/Sub Sharing
        print("\n[Phase 2] 🔬 Specialist Execution & Pub/Sub Blackboard Sharing...")
        for sp in self.specialists:
            finding = sp.execute(self.bb)
            notified = InterAgentCommunicator.broadcast(finding, self.bb)
            sub_note = f" (notified: {', '.join(notified)})" if notified else ""
            print(f"   {sp.icon} [{sp.name}] published {finding.id} ({finding.topic}){sub_note}")

        # Step 3: Event-Driven Stale Assumption Detection & Re-Execution
        print("\n[Phase 3] 🔄 Consistency Check & Stale Assumption Detection...")
        reg_f = self.bb.get_finding_by_topic("regulatory_clearance")
        if reg_f:
            delay = reg_f.numbers.get("expected_delay_weeks", 0)
            print(f"   • Propagating upstream constraint: timeline_shift_weeks = +{delay} weeks.")
            self.bb.set_constraint("timeline_shift_weeks", delay, increment_version=True)

        if self.bb.stale_finding_ids:
            stale_list = list(self.bb.stale_finding_ids)
            print(f"   ⚠️ STALE DETECTED: Finding(s) {stale_list} invalidated (Blackboard now at v{self.bb.assumptions_version})!")
            for sp in self.specialists:
                # Re-run any specialist whose previously published finding is now marked stale
                sp_stale = [f for f in self.bb.findings.values() if f.agent == sp.name and f.id in self.bb.stale_finding_ids]
                if sp_stale:
                    old_f = sp_stale[-1]
                    print(f"   -> Re-running {sp.name} under v{self.bb.assumptions_version} assumptions...")
                    new_f = sp.execute(self.bb)
                    InterAgentCommunicator.broadcast(new_f, self.bb)
                    if "net_margin_pct" in old_f.numbers and "net_margin_pct" in new_f.numbers:
                        print(
                            f"   ✓ {sp.name} updated [{new_f.id}]: Net Margin shifted "
                            f"{old_f.numbers['net_margin_pct']}% ➔ {new_f.numbers['net_margin_pct']}%."
                        )
                    else:
                        print(f"   ✓ {sp.name} refreshed finding [{new_f.id}] under v{self.bb.assumptions_version}.")
        else:
            print("   ✓ All findings consistent with current Blackboard assumptions version.")

        # Step 4: Conflict Detection, Negotiation & Escalation
        print("\n[Phase 4] 🤝 Conflict Detection, Peer Negotiation, & Escalation...")
        status = self.conflict_handler.negotiate(self.bb)

        if status == "UNRESOLVED" and self.escalation_handler:
            self.escalation_handler.escalate(self.bb)
        elif status == "RESOLVED":
            if not self.bb.decision_log:
                detail = self.bb.conflicts[-1].resolution_details if self.bb.conflicts else "Negotiated consensus achieved."
                self.bb.decision_log.append(f"Team negotiated consensus: {detail}")
        else:
            if not self.bb.decision_log:
                self.bb.decision_log.append("Clean consensus: All specialist constraints align without trade-offs.")

        # Step 5: Synthesis
        print("\n[Phase 5] 📑 Synthesizing Unified Traceable Recommendation...")
        return self._synthesize_clauses()

    def _synthesize_clauses(self) -> List[RecommendationClause]:
        mkt = self.bb.get_finding_by_topic("market_demand")
        reg = self.bb.get_finding_by_topic("regulatory_clearance")
        plan = self.bb.get_finding_by_topic("launch_timeline")
        fin = self.bb.get_finding_by_topic("financial_margin")
        comp = self.bb.get_finding_by_topic("competitive_landscape")

        # Domain-specific structured synthesis when standard Nova Devices findings exist
        if mkt and reg and fin and comp:
            clauses = [
                RecommendationClause(
                    clause_text=f"Market opportunity in {self.bb.goal['country']} represents a ${mkt.numbers.get('segment_size_m')}M addressable segment growing at {mkt.numbers.get('growth_pct')}% per year.",
                    source_agents=[mkt.agent],
                    source_findings=[mkt.id],
                    supporting_numbers={"segment_size_m": mkt.numbers.get('segment_size_m'), "growth_pct": mkt.numbers.get('growth_pct')}
                ),
                RecommendationClause(
                    clause_text=f"Mandatory regulatory filings require {reg.numbers.get('expected_delay_weeks')} weeks with statutory fees of ${reg.numbers.get('fee_usd'):,}.",
                    source_agents=[reg.agent],
                    source_findings=[reg.id],
                    supporting_numbers={"delay_weeks": reg.numbers.get('expected_delay_weeks'), "fees": reg.numbers.get('fee_usd')}
                ),
            ]
            if plan:
                clauses.append(RecommendationClause(
                    clause_text=f"Baseline GTM plan targets a {plan.numbers.get('target_weeks')}-week launch window with ${plan.numbers.get('ad_committed_usd'):,} in non-refundable ad commitments.",
                    source_agents=[plan.agent],
                    source_findings=[plan.id],
                    supporting_numbers={"target_weeks": plan.numbers.get('target_weeks'), "ad_committed_usd": plan.numbers.get('ad_committed_usd')}
                ))
            clauses.append(RecommendationClause(
                clause_text=f"Unit economics yield a landed unit cost of ${fin.numbers.get('landed_unit_cost')} with projected net margin of {fin.numbers.get('net_margin_pct')}%.",
                source_agents=[fin.agent],
                source_findings=[fin.id],
                supporting_numbers={"margin_pct": fin.numbers.get('net_margin_pct'), "cert_cost_cap": fin.numbers.get('cert_cost_cap')}
            ))

            final_action = self.bb.decision_log[-1] if self.bb.decision_log else "Proceed standard"
            lead_sources = [comp.agent]
            if self.bb.conflicts:
                lead_sources.extend(self.bb.conflicts[-1].involved_agents)

            clauses.append(RecommendationClause(
                clause_text=f"Executive Action: {final_action}",
                source_agents=list(dict.fromkeys(lead_sources)),
                source_findings=[comp.id, fin.id],
                supporting_numbers={
                    "competitor_gap_wks": comp.numbers.get('weeks_to_fill_gap'),
                    "resolved_status": self.bb.conflicts[-1].status if self.bb.conflicts else "CONSENSUS"
                }
            ))
            return clauses

        # Generic synthesis for any pluggable specialist roster
        clauses = []
        latest_by_topic: Dict[str, Finding] = {}
        for f in self.bb.findings.values():
            latest_by_topic[f.topic] = f

        for f in latest_by_topic.values():
            clauses.append(RecommendationClause(
                clause_text=f.claim,
                source_agents=[f.agent],
                source_findings=[f.id],
                supporting_numbers=f.numbers
            ))

        final_action = self.bb.decision_log[-1] if self.bb.decision_log else "Proceed per specialist consensus."
        all_agents = list(dict.fromkeys([f.agent for f in latest_by_topic.values()]))
        all_fids = [f.id for f in latest_by_topic.values()]
        clauses.append(RecommendationClause(
            clause_text=f"Executive Action: {final_action}",
            source_agents=all_agents,
            source_findings=all_fids,
            supporting_numbers={"status": self.bb.conflicts[-1].status if self.bb.conflicts else "CONSENSUS"}
        ))
        return clauses

def render_attribution_table(clauses: List[RecommendationClause]):
    print("\n" + "="*95)
    print("FINAL UNIFIED RECOMMENDATION — CLAUSE ATTRIBUTION TRACE")
    print("="*95)
    data = []
    for i, c in enumerate(clauses, 1):
        data.append({
            "Clause #": f"Clause {i}",
            "Synthesized Recommendation Text": c.clause_text,
            "Source Agent(s)": ", ".join(c.source_agents),
            "Finding ID(s)": ", ".join(c.source_findings),
            "Supporting Metrics": str(c.supporting_numbers)
        })
    df_table = pd.DataFrame(data)
    pd.set_option('display.max_colwidth', None)
    display(df_table)

print("✓ Domain-Agnostic CollabCouncil Orchestrator & Traceability Engine ready.")

✓ Domain-Agnostic CollabCouncil Orchestrator & Traceability Engine ready.


# 7. Scenario A Demo (NovaBand in Brazil)
- **Profile**: High demand (+40% vs baseline), 10-week ANVISA regulatory delay, cost cap ~$40K, no urgent competitor.
- **Expected Outcome**: Agents negotiate and self-resolve via a phased launch plan.

In [ ]:
# Cell 7: Interactive Execution of Scenarios
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load the scenarios dynamically
with open("/content/scenarios.json", "r") as f:
    scenarios = json.load(f)

# Extract unique products and countries from the dataset
products = sorted(list(set([s.get("product") for s in scenarios if s.get("product")])))
countries = sorted(list(set([s.get("country") for s in scenarios if s.get("country")])))

# Create Dropdowns and Button
product_dropdown = widgets.Dropdown(options=products, description='Product:')
country_dropdown = widgets.Dropdown(options=countries, description='Country:')
run_button = widgets.Button(description='Run Evaluation', button_style='primary')
output_area = widgets.Output()

# Define what happens when the button is clicked
def on_run_clicked(b):
    with output_area:
        clear_output(wait=True) # Clears the previous run's output
        p = product_dropdown.value
        c = country_dropdown.value

        print(f"\n{'*'*80}")
        print(f"EVALUATING SCENARIO: {p} in {c}")
        print(f"{'*'*80}\n")

        try:
            global orch
            global clauses
            orch = CollabCouncilOrchestrator(goal_product=p, goal_country=c)
            clauses = orch.run_pipeline()
            render_attribution_table(clauses)
        except Exception as e:
            print(f"❌ Failed to evaluate {p} in {c}: {str(e)}")

# Attach the click event to the button
run_button.on_click(on_run_clicked)

# Display the interactive UI elements
ui = widgets.HBox([product_dropdown, country_dropdown, run_button])
display(ui, output_area)

Output()

# 10. Communication Network & Sequence Visualizations
Visualizes peer-to-peer data sharing topology and timestamped sequence of interactions.

In [ ]:
# Cell 10: Network and Communication Visualizations
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

# Ensure a scenario has been run
if 'orch' not in globals():
    print("❌ No scenario data found. Please run a scenario in Cell 7 first.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    current_product = orch.bb.goal.get("product", "Unknown")
    current_country = orch.bb.goal.get("country", "Unknown")

    # 1. Network Graph of Inter-Agent Messages (Dynamic)
    G = nx.DiGraph()
    agent_colors = {
        "Market Research Specialist": "#3498db",
        "Regulatory & Compliance Specialist": "#e74c3c",
        "Financial Modeling Specialist": "#2ecc71",
        "Competitive Intelligence Specialist": "#9b59b6",
        "Marketing & Planning Specialist": "#f39c12"
    }

    for msg in orch.bb.message_log:
        G.add_edge(msg.sender, msg.receiver, label=msg.finding_id)

    pos = nx.spring_layout(G, seed=42, k=1.2)
    node_colors = [agent_colors.get(node, "#95a5a6") for node in G.nodes()]

    nx.draw_networkx_nodes(G, pos, ax=ax1, node_color=node_colors, node_size=2800, alpha=0.9)
    nx.draw_networkx_labels(G, pos, ax=ax1, font_size=8, font_weight="bold", font_family="sans-serif")
    nx.draw_networkx_edges(G, pos, ax=ax1, edge_color="#7f8c8d", arrows=True, arrowsize=18, width=1.8, connectionstyle="arc3,rad=0.1")

    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax1, font_size=7)
    ax1.set_title(f"Inter-Agent Information Sharing ({current_product} in {current_country})", fontsize=12, fontweight="bold")
    ax1.axis("off")

    # 2. Sequence Timeline Plot
    msgs = orch.bb.message_log
    if msgs:
        times = [m.timestamp - msgs[0].timestamp for m in msgs]
        senders = [m.sender.split()[0] for m in msgs]
        receivers = [m.receiver.split()[0] for m in msgs]
        labels = [f"{s} ➔ {r}\n({m.finding_id})" for s, r, m in zip(senders, receivers, msgs)]

        y_pos = np.arange(len(msgs))
        ax2.barh(y_pos, [t + 0.1 for t in times], color="#34495e", height=0.4, align="center")
        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(labels, fontsize=9)
        ax2.invert_yaxis()
        ax2.set_xlabel("Relative Timeline Offset (seconds)", fontsize=10)
        ax2.set_title("Sequence of Message Events on Blackboard", fontsize=12, fontweight="bold")
        ax2.grid(axis='x', linestyle='--', alpha=0.6)
    else:
        ax2.text(0.5, 0.5, "No inter-agent messages recorded", ha='center')

    plt.tight_layout()
    plt.show()

❌ No scenario data found. Please run a scenario in Cell 7 first.


# 11. Single-LLM Generalist vs. CollabCouncil Multi-Agent Comparison
Side-by-side comparison demonstrating the difference between generic zero-shot reasoning from a single LLM and grounded, tool-backed multi-agent deliberation.

In [ ]:
# Cell 11: Single Generalist LLM vs. CollabCouncil Comparison
import pandas as pd

def run_single_generalist(product: str, country: str) -> str:
    prompt = f"Goal: Should Nova Devices launch {product} in {country} next quarter? Give a direct business recommendation in 2 sentences."
    system = "You are a general business AI. Provide an executive recommendation without access to private internal company databases or specific regulatory files."
    return llm(system, prompt)

if 'orch' not in globals() or 'clauses' not in globals():
    print("❌ No scenario data found. Please run a scenario in Cell 7 first.")
else:
    current_product = orch.bb.goal.get("product", "Unknown")
    current_country = orch.bb.goal.get("country", "Unknown")

    print(f"Comparing Single LLM vs CollabCouncil for: {current_product} in {current_country}...\n")

    single_llm_output = run_single_generalist(current_product, current_country)
    team_output = " ".join([c.clause_text for c in clauses])

    comp_df = pd.DataFrame([
        {
            "Evaluation Dimension": "Numeric Grounding",
            "Single Generalist LLM": "None (gives generic strategic advice or hallucinated estimates).",
            "CollabCouncil Multi-Agent System": "Exact numbers dynamically pulled from external datasets."
        },
        {
            "Evaluation Dimension": "Conflict Handling",
            "Single Generalist LLM": "Fails to identify precise timing mismatches or calculate exact margin deficits.",
            "CollabCouncil Multi-Agent System": "Explicit rule checks flag margin deficits & competitor preemption; executes structured negotiation."
        },
        {
            "Evaluation Dimension": "Traceability",
            "Single Generalist LLM": "Opaque single-pass generation; no audit trail for claims.",
            "CollabCouncil Multi-Agent System": "100% auditable: every clause linked to finding IDs and specialist agent authors."
        },
        {
            "Evaluation Dimension": "Decision Output",
            "Single Generalist LLM": single_llm_output,
            "CollabCouncil Multi-Agent System": team_output
        }
    ])

    pd.set_option('display.max_colwidth', None)
    display(comp_df)

❌ No scenario data found. Please run a scenario in Cell 7 first.


# 12. Pluggable Specialist Roster Demo (Architecture Reusability)
Demonstrates the domain-agnostic nature of the coordination core by solving an IT architecture goal:
*"Should we migrate our monolith to microservices?"* with 3 custom specialists: Security, DevOps/Cost, and Delivery-Capacity.

In [ ]:
# Cell 12: Pluggable Specialist Roster Demo for Monolith vs Microservices Decision
# Uses the exact same CollabCouncilOrchestrator & Attribution Table as the Product Launch domain!

class TechSpecialist(BaseSpecialist):
    def __init__(self, name: str, domain: str, icon: str, dataset_file: str, subscribes_to: Optional[List[str]] = None):
        super().__init__(name, domain, icon, subscribes_to=subscribes_to)
        self.dataset_file = dataset_file

    def get_data(self) -> Dict[str, Any]:
        with open(os.path.join(DATA_DIR, self.dataset_file), "r") as f:
            return json.load(f)

    def execute(self, bb: Blackboard) -> Finding:
        data = self.get_data()
        claim = llm(
            system_instruction=f"You are the {self.name}. Summarize your evaluation in 1 clear sentence incorporating key metrics.",
            prompt=f"Domain: {self.domain}, Metrics: {data}."
        )
        return Finding(
            id=f"F_{self.name[:3].upper()}_01",
            agent=self.name,
            topic=self.domain.lower().replace(" ", "_").replace("/", "_"),
            claim=claim,
            numbers=data,
            assumptions_version=bb.assumptions_version,
            confidence=0.90
        )

class TechConflictHandler:
    """Domain-specific conflict rules plugged into the generic CollabCouncilOrchestrator."""
    @staticmethod
    def negotiate(bb: Blackboard) -> str:
        sec_f = bb.get_finding_by_topic("identity_&_attack_surface")
        dev_f = bb.get_finding_by_topic("sprint_velocity_&_team_readiness")
        cost_f = bb.get_finding_by_topic("cloud_infrastructure_tco")

        sec_data = sec_f.numbers if sec_f else {}
        dev_data = dev_f.numbers if dev_f else {}
        cost_data = cost_f.numbers if cost_f else {}

        required_eng_weeks = sec_data.get("hardening_baseline_eng_weeks", 0) + (
            cost_data.get("total_services", 0)
            * (dev_data.get("migration_eng_weeks_per_service", 0) + sec_data.get("per_service_security_eng_weeks", 0))
        )
        projected_monthly_cost = cost_data.get("current_infra_monthly_usd", 0) + (
            cost_data.get("total_services", 0) * cost_data.get("incremental_infra_monthly_per_service_usd", 0)
        )
        spare_weeks = dev_data.get("spare_capacity_eng_weeks_this_quarter", 0)
        budget_cap = cost_data.get("infra_budget_cap_monthly_usd", float("inf"))

        if "mTLS" in sec_data.get("hard_rule", "") and required_eng_weeks > spare_weeks:
            print(f"\n⚡ [Conflict Detection] Security + Delivery capacity gap: {required_eng_weeks} eng-weeks required vs {spare_weeks} available.")
            proposal = llm(
                system_instruction="You are the Engineering Lead Mediator. Propose a 1-sentence phased migration compromise.",
                prompt=f"Required engineering weeks: {required_eng_weeks}, Available capacity: {spare_weeks}, Hard rule: {sec_data.get('hard_rule')}."
            )
            print(f"   🤖 [Mediator Proposal]: {proposal}")
            resolution = "Defer full migration: Execute security baseline and mTLS rollout this quarter before service extraction."
            bb.conflicts.append(ConflictRecord(
                conflict_id="CONF_TECH_CAP_01",
                issue=f"Engineering capacity deficit ({required_eng_weeks}w required > {spare_weeks}w available)",
                involved_agents=["Security Specialist", "Delivery Capacity Specialist", "DevOps Cost Specialist"],
                hard_constraints_violated=[],
                rounds_run=1,
                status="RESOLVED",
                resolution_details=resolution
            ))
            bb.decision_log.append(resolution)
            return "RESOLVED"

        if projected_monthly_cost > budget_cap:
            resolution = f"Halt migration: Projected cost (${projected_monthly_cost:,}/mo) exceeds budget cap (${budget_cap:,}/mo)."
            bb.conflicts.append(ConflictRecord(
                conflict_id="CONF_TECH_COST_01",
                issue=f"Monthly TCO (${projected_monthly_cost:,}) exceeds cap (${budget_cap:,})",
                involved_agents=["DevOps Cost Specialist", "Delivery Capacity Specialist"],
                hard_constraints_violated=[],
                rounds_run=1,
                status="RESOLVED",
                resolution_details=resolution
            ))
            bb.decision_log.append(resolution)
            return "RESOLVED"

        return "NO_CONFLICT"

tech_roster = [
    TechSpecialist("Security Specialist", "Identity & Attack Surface", "🔒", "security.json"),
    TechSpecialist("DevOps Cost Specialist", "Cloud Infrastructure TCO", "☁️", "devops_cost.json", subscribes_to=["identity_&_attack_surface"]),
    TechSpecialist("Delivery Capacity Specialist", "Sprint Velocity & Team Readiness", "⚡", "delivery.json", subscribes_to=["identity_&_attack_surface"])
]

tech_orch = CollabCouncilOrchestrator(
    goal={"title": "Migrate Monolith to Microservices in Q3", "decision": "Migrate Monolith to Microservices", "timeframe": "Q3"},
    specialists=tech_roster,
    conflict_handler=TechConflictHandler
)

tech_clauses = tech_orch.run_pipeline()
render_attribution_table(tech_clauses)


🚀 INITIATING COLLABCOUNCIL PIPELINE: Migrate Monolith to Microservices in Q3

[Phase 1] 📋 Decomposing High-Level Goal & Assigning Workstreams...
   🤖 Assigned [Security Specialist] -> Assess identity management and attack surface reduction for the microservices migration.
   🤖 Assigned [DevOps Cost Specialist] -> Calculate cloud infrastructure TCO and propose cost‑optimization strategies for the new microservices.
   🤖 Assigned [Delivery Capacity Specialist] -> Evaluate sprint velocity and team readiness to schedule phased migration within Q3.

[Phase 2] 🔬 Specialist Execution & Pub/Sub Blackboard Sharing...
   🔒 [Security Specialist] published F_SEC_01 (identity_&_attack_surface) (notified: DevOps Cost Specialist, Delivery Capacity Specialist)
   ☁️ [DevOps Cost Specialist] published F_DEV_01 (cloud_infrastructure_tco)
   ⚡ [Delivery Capacity Specialist] published F_DEL_01 (sprint_velocity_&_team_readiness)

[Phase 3] 🔄 Consistency Check & Stale Assumption Detection...
   ✓ All findi

,Clause #,Synthesized Recommendation Text,Source Agent(s),Finding ID(s),Supporting Metrics
0,Clause 1,"The identity and attack‑surface hardening will take 8 weeks for the baseline plus 1 week per service, must enforce mTLS and a secrets manager before any service goes live, and must address cross‑service auth exposure, automate secret rotation, and implement mesh egress policies.",Security Specialist,F_SEC_01,"{'hardening_baseline_eng_weeks': 8, 'per_service_security_eng_weeks': 1, 'hard_rule': 'mTLS and a secrets manager must be in place before any service goes live', 'risk_notes': ['Cross-service communication increases auth attack surface', 'Automated secret rotation required across microservice cluster', 'Service mesh egress policy enforcement needed']}"
1,Clause 2,"With 10 services, the projected monthly infrastructure cost of $12,000 + (10 × $900) = $21,000 exceeds the $17,000 budget cap by $4,000.",DevOps Cost Specialist,F_DEV_01,"{'total_services': 10, 'current_infra_monthly_usd': 12000, 'incremental_infra_monthly_per_service_usd': 900, 'infra_budget_cap_monthly_usd': 17000}"
2,Clause 3,"With 30 spare engineering weeks this quarter we can theoretically complete up to six 5‑week migrations, yet the 10‑week release deadline and core‑platform modernization commitments sharply limit usable capacity, leaving only minimal slack for additional work.",Delivery Capacity Specialist,F_DEL_01,"{'spare_capacity_eng_weeks_this_quarter': 30, 'migration_eng_weeks_per_service': 5, 'release_deadline_weeks': 10, 'roadmap_commitments_note': 'Core platform modernization commitments limit available engineering capacity.'}"
3,Clause 4,Executive Action: Defer full migration: Execute security baseline and mTLS rollout this quarter before service extraction.,"Security Specialist, DevOps Cost Specialist, Delivery Capacity Specialist","F_SEC_01, F_DEV_01, F_DEL_01",{'status': 'RESOLVED'}


# 13. System Summary & Scenario Comparison Matrix

In [ ]:
# Cell 13: Dynamic Summary Evaluation Table for the Current Scenario

import pandas as pd

# Ensure a scenario has been run
if 'orch' not in globals():
    print("❌ No scenario data found. Please run a scenario in Cell 7 first.")
else:
    current_product = orch.bb.goal.get("product", "Unknown")
    current_country = orch.bb.goal.get("country", "Unknown")

    # Determine conflict details dynamically from the Blackboard
    if orch.bb.conflicts:
        primary_conflict = orch.bb.conflicts[0].issue
        rounds_run = orch.bb.conflicts[-1].rounds_run
        outcome_status = orch.bb.conflicts[-1].status
        resolution_details = orch.bb.conflicts[-1].resolution_details

        if outcome_status == "ESCALATED":
            escalated = "Yes (Irreconcilable Trade-Off)"
        elif outcome_status == "UNRESOLVED":
            escalated = "Yes (Deadlock)"
        else:
            escalated = "No (Self-Resolved via System Negotiation)"
    else:
        primary_conflict = "None (Constraints and margins align)"
        rounds_run = 0
        outcome_status = "CONSENSUS"
        escalated = "No (Direct Approval)"
        resolution_details = "Proceed standard"

    summary_data = [
        {
            "Scenario": f"{current_product} in {current_country}",
            "Primary Conflict": primary_conflict,
            "Rounds": rounds_run,
            "Outcome Status": outcome_status,
            "Escalated to Human?": escalated,
            "Key Condition": resolution_details
        }
    ]

    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

❌ No scenario data found. Please run a scenario in Cell 7 first.


# 14. Limitations & Future Work

1. **Rule-Based Conflict Triggers**: Conflict predicates (e.g. `timeline_shift > target` or `margin < 0`) are codified in explicit deterministic Python logic. In an enterprise system, this can be combined with semantic conflict extractors.
2. **Synthetic Data Realism**: Numbers reflect realistic structural proportions but are illustrative models rather than dynamic feeds from ERP/MES or live regulatory gazettes.
3. **Turn-Based Coordination**: The current protocol uses synchronous rounds; an enterprise production blackboard would benefit from asynchronous pub/sub queues (e.g. Redis, Kafka) supporting distributed transaction locking.